# Individual Jupyter Notebook
## Student Name: Jiamu Zheng SID:530836081
### Task1
Data Cleaning

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

current_dir = Path.cwd()
data_path = current_dir / "data" / "Region summary_ New South Wales STE 1.csv"

df = pd.read_csv(data_path)

task1_df = df.rename(
    columns={
        "Measure Code": "measure_code",
        "Parent Description": "parent_description",
        "Description": "description",
    }
).copy()

task1_df.columns = [f"y_{col}" if str(col).isdigit() else col for col in task1_df.columns]
year_cols = [col for col in task1_df.columns if col.startswith("y_")]

task1_df["unit"] = task1_df["description"].str.extract(r"\(([^()]*)\)\s*$")
task1_df["description_clean"] = task1_df["description"].str.replace(r"\s*\([^()]*\)\s*$", "", regex=True)
task1_df["non_null_years"] = task1_df[year_cols].notna().sum(axis=1)
task1_df = task1_df[task1_df["non_null_years"] > 0].copy()

task1_long_df = task1_df.melt(
    id_vars=["measure_code", "parent_description", "description_clean", "unit"],
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)

task1_long_df = task1_long_df.dropna(subset=["value"]).copy()
task1_long_df["year"] = task1_long_df["year"].str.replace("y_", "").astype(int)
task1_analysis_df = (
    task1_long_df[task1_long_df["year"] != 2025]
    .drop_duplicates(subset=["measure_code", "description_clean", "year"])
    .copy()
)

cleaning_summary_df = pd.DataFrame(
    {
        "item": [
            "original_rows",
            "rows_after_drop_all_null_years",
            "long_table_rows",
            "analysis_rows_without_2025",
            "rows_with_only_one_non_null_year",
            "non_null_values_in_2025",
        ],
        "value": [
            len(df),
            len(task1_df),
            len(task1_long_df),
            len(task1_analysis_df),
            int((task1_df["non_null_years"] == 1).sum()),
            int(task1_long_df[task1_long_df["year"] == 2025].shape[0]),
        ],
    }
)


### Income related statistics

Our group selected four broad themes for the statistical analysis: population, income, age, and housing. In this individual notebook, the focus is on income category, and 5 income-related statistics will be present in the following section

### Statistic 1: Mean-to-Median Income Ratio


In [3]:
# Step 1: Identify income indicators related to total income
income_total_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "total income",
        case=False,
        na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(income_total_check)

# Step 2: Select the confirmed mean and median total income indicators
income_ratio_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin(["INCOME_17", "INCOME_36"])
][
    ["measure_code", "year", "value"]
].copy()

# Step 3: Reshape so each year has both median and mean income
income_ratio_wide = income_ratio_df.pivot(
    index="year",
    columns="measure_code",
    values="value"
).reset_index()

# Step 4: Rename columns based on the checked descriptions
income_ratio_wide = income_ratio_wide.rename(
    columns={
        "INCOME_17": "median_total_income",
        "INCOME_36": "mean_total_income"
    }
)

# Step 5: Calculate mean-to-median income ratio
income_ratio_wide["mean_to_median_ratio"] = (
    income_ratio_wide["mean_total_income"] /
    income_ratio_wide["median_total_income"]
)

display(income_ratio_wide)

,measure_code,parent_description,description_clean,unit
3638,INCOME_17,Personal income in Australia - year ended 30 June,Median total income (excl. Government pensions...,$
3637,INCOME_18,Personal income in Australia - year ended 30 June,Total income (excl. Government pensions and al...,$m
3635,INCOME_19,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pension...,no.
3636,INCOME_35,Personal income in Australia - year ended 30 June,Total income earners (excl. Government pension...,years
3639,INCOME_36,Personal income in Australia - year ended 30 June,Mean total income (excl. Government pensions a...,$


measure_code,year,median_total_income,mean_total_income,mean_to_median_ratio
0,2018,47852.0,64528.0,1.348491
1,2019,49522.0,66310.0,1.339001
2,2020,50567.0,67677.0,1.338363
3,2021,53905.0,71919.0,1.334181
4,2022,55105.0,75511.0,1.370311


This statistic was calculated to assess the degree of income inequality. Comparing mean income with median income provides a simple way to determine whether average income is being disproportionately influenced by high-income earners.

The mean-to-median income ratio was calculated by dividing mean total income (INCOME_36) by median total income (INCOME_17) for each available year. 

The ratio remained consistently above 1, with values ranging from approximately 1.33 to 1.37. This indicates that mean total income was around 33–37% higher than median total income.

Because the mean is more sensitive to extremely high values, a ratio substantially greater than 1 suggests that higher-income earners are pulling the average income above the level earned by a typical individual. This provides evidence that the income distribution is positively skewed and that income inequality exists.


### Statistics 2: Income Growth Rate

In [4]:
#Income growth rate
income_growth_df = task1_analysis_df[
    task1_analysis_df["measure_code"] == "INCOME_17"
].copy()


income_growth_df = income_growth_df[
    ["year", "value"]
].rename(
    columns={"value": "median_total_income"}
)


income_growth_df = income_growth_df.sort_values("year")


income_growth_df["income_growth_rate"] = (
    income_growth_df["median_total_income"]
    .pct_change() * 100
)

display(income_growth_df)

,year,median_total_income,income_growth_rate
3638,2018,47852.0,NaN
4438,2019,49522.0,3.489927
5238,2020,50567.0,2.110173
6038,2021,53905.0,6.601143
6838,2022,55105.0,2.226139


This statistic was calculated to measure how the income of a typical individual changed over time. It complements the previous statistic by focusing on income growth rather than income inequality.

The growth rate was calculated using median total income (INCOME_17), which is less affected by extreme values and therefore provides a better representation of typical income.

The results show that median total income increased from $47,852 in 2018 to $55,105 in 2022, representing an overall increase of approximately 15.2%. The annual growth rates were 2.8% in 2019, 1.9% in 2020, 6.6% in 2021, and 3.6% in 2022.


### Statistics 3: Investment Income Premium

In [5]:

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
# Step 1: Explore all mean income indicators
mean_income_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "Mean", case=False, na=False
    ) &
    task1_analysis_df["description_clean"].str.contains(
        "income", case=False, na=False
    ) &
    (task1_analysis_df["unit"] == "$")
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(mean_income_check)

# Step 2: Based on the table above, select:
# INCOME_21 = Mean employee income
# INCOME_27 = Mean investment income

investment_premium_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin(["INCOME_21", "INCOME_27"])
][
    ["measure_code", "year", "value"]
].copy()

# Step 3: Reshape so each year has both income types
investment_premium_wide = investment_premium_df.pivot(
    index="year",
    columns="measure_code",
    values="value"
).reset_index()

# Step 4: Rename columns based on the identified indicators
investment_premium_wide = investment_premium_wide.rename(columns={
    "INCOME_21": "mean_employee_income",
    "INCOME_27": "mean_investment_income"
})

# Step 5: Calculate investment income premium
investment_premium_wide["investment_income_premium"] = (
    investment_premium_wide["mean_investment_income"] /
    investment_premium_wide["mean_employee_income"]
)

display(investment_premium_wide)

,measure_code,parent_description,description_clean,unit
3615,INCOME_21,Personal income in Australia - year ended 30 June,Mean employee income,$
3621,INCOME_24,Personal income in Australia - year ended 30 June,Mean own unincorporated business income,$
3627,INCOME_27,Personal income in Australia - year ended 30 June,Mean investment income,$
3633,INCOME_30,Personal income in Australia - year ended 30 June,Mean superannuation and annuity income,$
3639,INCOME_36,Personal income in Australia - year ended 30 June,Mean total income (excl. Government pensions and allowances),$


measure_code,year,mean_employee_income,mean_investment_income,investment_income_premium
0,2018,64509.0,9525.0,0.147654
1,2019,66377.0,9698.0,0.146105
2,2020,68690.0,9373.0,0.136454
3,2021,71848.0,10133.0,0.141034
4,2022,74232.0,12229.0,0.164740


This statistic was calculated to compare mean investment income with mean employee income. It provides an indication of how significant investment income is relative to wages and salaries.

The investment income premium was calculated by dividing mean investment income by mean employee income for each available year. 

The ratio ranged from approximately 0.14 to 0.16. This indicates that mean investment income was only around 14–16% of mean employee income.This result suggest that, on average, investment income represents a relatively small share of total income compared with wages and salaries and the employee income remains the dominant source of income for most individuals.


### Statistics 4: High-income share vs low-income share

In [6]:
# Step 1: Explore personal income bracket indicators
personal_income_check = task1_analysis_df[
    task1_analysis_df["parent_description"].str.contains(
        "Total personal income", case=False, na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(personal_income_check)

# Step 2: Select low-income and high-income brackets
income_share_df = task1_analysis_df[
    task1_analysis_df["measure_code"].isin([
        "PERSINC_2",  # $1-$499 per week
        "PERSINC_5",  # $2000-$2999 per week
        "PERSINC_6",  # $3000 or more per week
        "PERSINC_7"   # Nil income
    ])
].copy()

# Step 3: Classify selected indicators into low-income and high-income groups
income_share_df["income_group"] = income_share_df["measure_code"].map({
    "PERSINC_2": "low_income",
    "PERSINC_7": "low_income",
    "PERSINC_5": "high_income",
    "PERSINC_6": "high_income"
})

# Step 4: Sum the percentages within each income group by year
income_share_summary = (
    income_share_df
    .groupby(["year", "income_group"])["value"]
    .sum()
    .reset_index()
)

# Step 5: Reshape so each year has both low-income and high-income shares
income_share_wide = income_share_summary.pivot(
    index="year",
    columns="income_group",
    values="value"
).reset_index()

# Step 6: Calculate high-to-low income ratio
income_share_wide["high_to_low_income_ratio"] = (
    income_share_wide["high_income"] /
    income_share_wide["low_income"]
)

display(income_share_wide)

,measure_code,parent_description,description_clean,unit
2329,INCMIG_2,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$1-$499 per week,%
2330,INCMIG_3,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$500-$999 per week,%
2331,INCMIG_4,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$1000-$1999 per week,%
2332,INCMIG_5,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$2000-$2999 per week,%
2333,INCMIG_6,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,$3000 or more per week,%
2334,INCMIG_7,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Nil income,%
2335,INCMIG_8,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Negative income,%
2336,INCMIG_9,Total personal income (weekly) of persons born overseas - Persons aged 15 years and over - Census,Income inadequately described or not stated,%
1991,PERSINC_2,Total personal income (weekly) - Persons aged 15 years and over - Census,$1-$499 per week,%
1992,PERSINC_3,Total personal income (weekly) - Persons aged 15 years and over - Census,$500-$999 per week,%


income_group,year,high_income,low_income,high_to_low_income_ratio
0,2016,8.8,37.0,0.237838
1,2021,13.6,31.2,0.435897


This statistic was calculated to compare the proportion of individuals in selected high-income and low-income groups. It provides a simple measure of the overall income distribution.

The high-income group was defined as individuals earning $2,000–$2,999 per week and $3,000 or more per week. The low-income group was defined as individuals with nil income and those earning $1–$499 per week. The high-to-low income ratio was calculated by dividing the combined high-income share by the combined low-income share.

The results show that the high-income share was substantially smaller than the low-income share in both available years. The high-to-low income ratio was 0.24 in 2016 to 0.44 in 2021.This indicate that low-income earners remained more prevalent than high-income earners. However, the increase in the ratio suggests that the proportion of high-income earners grew relative to the proportion of low-income earners over time.


### Statistic 5: Employee income as main source of income

In [7]:
# Step 1: Explore income source indicators
income_source_check = task1_analysis_df[
    task1_analysis_df["description_clean"].str.contains(
        "main source of income", case=False, na=False
    )
][
    ["measure_code", "parent_description", "description_clean", "unit"]
].drop_duplicates().sort_values("measure_code")

display(income_source_check)

# Step 2: Select employee income as main source of income
employee_income_source_df = task1_analysis_df[
    task1_analysis_df["measure_code"] == "INCOME_22"
][
    ["year", "value"]
].rename(
    columns={"value": "employee_income_main_source_share"}
).copy()

employee_income_source_df = employee_income_source_df.sort_values("year")

display(employee_income_source_df)

,measure_code,parent_description,description_clean,unit
3616,INCOME_22,Personal income in Australia - year ended 30 June,Employee income as main source of income,%
3622,INCOME_25,Personal income in Australia - year ended 30 June,Own unincorporated business income as main source of income,%
3628,INCOME_28,Personal income in Australia - year ended 30 June,Investment income as main source of income,%
3634,INCOME_31,Personal income in Australia - year ended 30 June,Superannuation and annuity income as main source of income,%


,year,employee_income_main_source_share
3616,2018,79.8
4416,2019,79.7
5216,2020,79.7
6016,2021,80.0
6816,2022,80.4


This statistic was calculated to measure the proportion of individuals whose main source of income was employee income. It provides insight into how dependent the population is on wages and salaries relative to other income sources such as investments, business income, and government transfers.

The statistic was based on the indicator Employee income as main source of income, which is reported as a percentage of all income earners.

The results show that this proportion remained consistently high, ranging from 79.7% to 80.4% between 2018 and 2022. It suggest that employee income remains the dominant source of income for the vast majority of Australians, while non-employment income sources play a secondary role for most individuals.



### Task2

The selected SA4 code is 120, which corresponds to Sydney - Inner West. This section identifies all SA2 regions within this SA4 and collects Points of Interest (POI) records for these SA2s using the helper functions in `task2_sa2.py`.

The file `task2_sa2.py` is used as a helper script for Task 2 and Task 3. It contains reusable functions for reading SA2 and SA4 shapefile data, identifying SA2 regions within a selected SA4, collecting POI records from the NSW POI API, filtering POIs by SA2 polygon, saving the results into a SQLite database, and calculating the final well-resourced score.

The workflow first retrieves the SA2 boundary and bounding box information for SA4 120. It then queries the NSW POI API for each SA2 area and filters the returned POIs so that only points located inside the relevant SA2 polygon are retained. The final outputs are an SA2 boundary table, a POI table, and a POI count table for each SA2.

In [2]:
import importlib
import task2_sa2
importlib.reload(task2_sa2)

SA4_CODE = "120"
AREA_NAME = "area_1"

sa2_bbox_df = task2_sa2.get_sa2_bbox_df_by_code(SA4_CODE)
sa2_bbox_df["area_name"] = AREA_NAME

display(sa2_bbox_df[["area_name", "sa4_code", "sa4_name", "sa2_main", "sa2_name"]])

poi_df = task2_sa2.get_sa4_poi_df_by_code(SA4_CODE)
poi_df["area_name"] = AREA_NAME

display(poi_df.head())

poi_count_df = (
    poi_df
    .groupby(["sa2_main", "sa2_name"])
    .size()
    .reset_index(name="poi_count")
    .sort_values("poi_count", ascending=False)
)

display(poi_count_df)

,area_name,sa4_code,sa4_name,sa2_main,sa2_name
0,area_1,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita
1,area_1,120,Sydney - Inner West,120011385,Drummoyne - Rodd Point
2,area_1,120,Sydney - Inner West,120011386,Five Dock - Abbotsford
3,area_1,120,Sydney - Inner West,120011672,Concord West - North Strathfield
4,area_1,120,Sydney - Inner West,120011673,Rhodes
5,area_1,120,Sydney - Inner West,120021387,Balmain
6,area_1,120,Sydney - Inner West,120021389,Lilyfield - Rozelle
7,area_1,120,Sydney - Inner West,120021674,Annandale (NSW)
8,area_1,120,Sydney - Inner West,120021675,Leichhardt
9,area_1,120,Sydney - Inner West,120031392,Canterbury (North) - Ashbury


,objectid,poiname,poitype,poigroup,longitude,latitude,sa4_code,sa4_name,sa2_main,sa2_name,area_name
0,849,NaN,Place Of Worship,1,151.109531,-33.860946,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
1,852,NaN,Place Of Worship,1,151.100483,-33.861704,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
2,1120,CONCORD GOLF COURSE,Golf Course,3,151.097332,-33.850397,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
3,1121,PRINCE EDWARD PARK,Park,3,151.119009,-33.852815,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1
4,1122,MASSEY PARK GOLF COURSE,Golf Course,3,151.111647,-33.853167,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,area_1


,sa2_main,sa2_name,poi_count
1,120011385,Drummoyne - Rodd Point,267
6,120021389,Lilyfield - Rozelle,235
8,120021675,Leichhardt,192
5,120021387,Balmain,152
12,120031395,Haberfield - Summer Hill,121
0,120011383,Concord - Mortlake - Cabarita,113
2,120011386,Five Dock - Abbotsford,89
17,120031678,Burwood (NSW),87
10,120031393,Croydon Park - Enfield,79
13,120031396,Homebush,63


In [3]:
sa2_bbox_df = sa2_bbox_df[
    [
        "area_name",
        "sa4_code",
        "sa4_name",
        "sa2_main",
        "sa2_name",
        "state_name",
        "area_sqkm",
        "bbox_xmin",
        "bbox_ymin",
        "bbox_xmax",
        "bbox_ymax",
    ]
]

poi_df = poi_df[
    [
        "objectid",
        "poiname",
        "poitype",
        "poigroup",
        "longitude",
        "latitude",
        "area_name",
        "sa4_code",
        "sa4_name",
        "sa2_main",
        "sa2_name",
    ]
]

DB_PATH = task2_sa2.DATA_DIR / "task2_poi_SA4_120.db"

task2_sa2.save_to_sqlite(
    sa2_bbox_df,
    poi_df,
    str(DB_PATH)
)

print("Saved SA4 120 database to:", DB_PATH)

Saved SA4 120 database to: /Users/mac/Desktop/DA/Data2001/Group assignment/DATA2001_G56/data/task2_poi_SA4_120.db


The SA2 boundary table and POI table were saved into a local SQLite database. The `save_to_sqlite()` function also creates the score table used in Task 3.

### Task 3

The well-resourced score is calculated using the POI count for each SA2. A higher POI count suggests that an SA2 has more recorded local facilities and services.

In this workflow, the score calculation is implemented inside `task2_sa2.py`. When the SA2 and POI tables are saved using `save_to_sqlite()`, the helper function automatically creates the `sa2_scores` table. This table counts POIs for each SA2, standardises the counts using a z-score, and applies a sigmoid transformation to produce a final score between 0 and 1.

In [4]:
scores_df = task2_sa2.read_sql(
    """
    SELECT *
    FROM sa2_scores
    ORDER BY score DESC
    """,
    DB_PATH
)

display(scores_df)

,area_name,sa4_code,sa4_name,sa2_main,sa2_name,area_sqkm,poi_count,z_poi,score
0,area_1,120,Sydney - Inner West,120011385,Drummoyne - Rodd Point,3.7047,267,2.607473,0.931341
1,area_1,120,Sydney - Inner West,120021389,Lilyfield - Rozelle,3.6039,235,2.133131,0.894082
2,area_1,120,Sydney - Inner West,120021675,Leichhardt,2.9838,192,1.495732,0.816937
3,area_1,120,Sydney - Inner West,120021387,Balmain,2.5815,152,0.902804,0.711525
4,area_1,120,Sydney - Inner West,120031395,Haberfield - Summer Hill,3.4746,121,0.443285,0.609041
5,area_1,120,Sydney - Inner West,120011383,Concord - Mortlake - Cabarita,6.3199,113,0.324699,0.580469
6,area_1,120,Sydney - Inner West,120011386,Five Dock - Abbotsford,4.6202,89,-0.031058,0.492236
7,area_1,120,Sydney - Inner West,120031678,Burwood (NSW),1.9050,87,-0.060705,0.484829
8,area_1,120,Sydney - Inner West,120031393,Croydon Park - Enfield,3.9711,79,-0.179290,0.455297
9,area_1,120,Sydney - Inner West,120031396,Homebush,3.8432,63,-0.416462,0.397364
